installing necessary libraries

In [ ]:
!pip install opencv-python

importing all the necessary libraries needed for the work

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from PIL import Image
import cv2
import numpy as np
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from torch.utils.data import DataLoader
import torch
from torch.utils.data import Dataset

defining the functions needed for the pre-processing of the images

In [ ]:
def grayscale(image):
  return cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

In [ ]:
#deskewing the image that we got

def getSkewAngle(cvImage) -> float:
  # Prep image, copy, convert to gray scale, blur, and threshold
  newImage = cvImage.copy()
  #gray = cv2.cvtColor(newImage, cv2.COLOR_BGR2GRAY)
  blur = cv2.GaussianBlur(newImage, (9, 9), 0)
  thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

  kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (30, 5))
  dilate = cv2.dilate(thresh, kernel, iterations=10)

  contours, hierarchy= cv2.findContours(dilate, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
  contours = sorted(contours, key = cv2.contourArea, reverse = True)
  for c in contours:
    rect = cv2.boundingRect(c)
    x,y,w,h = rect
    cv2.rectangle(newImage,(x,y),(x+w,y+h),(0,255,0),2)

  largestContour= contours[0]
  print(len(contours))
  minAreaRect = cv2.minAreaRect(largestContour)
  cv2.imwrite('mesia/boxes.jpg',newImage)

  angle = minAreaRect[-1]
  if angle < -45:
    angle = 90 + angle
  return -1.0 * angle

def rotateImage(cvImage, angle:float):
  newImage= cvImage.copy()
  (h, w) = newImage.shape[:2]
  center= (w // 2, h // 2)
  M = cv2.getRotationMatrix2D(center, angle, 1.0)
  newImage= cv2.warpAffine(newImage, M, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
  return newImage

def deskew(cvImage):
  angle = getSkewAngle(cvImage)
  return rotateImage(cvImage, -1.0 * angle)

testing the pre-processing on a simple example with only one simple img

In [ ]:
#img pre-processing
#transforming the image to arrays using cv2

img_path= '/media/test1.jpg'
image=cv2.imread(img_path)

img_gray=grayscale(image)
cv2.imwrite("/media/gray.jpg",img_gray)

thresh, image_bw= cv2.threshold(img_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
#thresh return the threshold value used

cv2.imwrite("/media/bw.jpg",image_bw)

#we need to convert the image to PIL format so we can give it to the model
final_img=Image.open("/media/bw.jpg")

loading the model I'm using for this project: microsoft/trocr-small-handwritten and testing it on the simple img pre-processed before

In [ ]:
#the ocr model

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-small-handwritten')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-small-handwritten')
pixel_values = processor(images=final_img, return_tensors="pt").pixel_values

generated_ids = model.generate(pixel_values)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

printing the output and comparing it to the real text written in the img

In [ ]:
print(generated_text)

clean old Papadiert's from watch


adding more pre-processing to the img to improve the quality of the output by defining a new function: noise removal

In [ ]:
def noise_removal(image):

  #SHARPEN: This defines the edges so they don't bleed together
  kernel_sharpen = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
  image = cv2.filter2D(image, -1, kernel_sharpen)

  #a kernel is a small matrix that slides over your image to perform an operation
  kernel_closing = np.ones((3,1), np.uint8)

  #closing small holes
  image = cv2.morphologyEx(image, cv2.MORPH_CLOSE,kernel_closing)

  #blur with 3x3 area

  return (image)

applying the new function on the img

In [ ]:
no_noise= noise_removal(image_bw)
cv2.imwrite('/media/no_noise.jpg', no_noise)

True

In [ ]:
final_img=Image.open("/media/no_noise.jpg")

testing the same model on the new img saved after the noise removal

In [ ]:
#the ocr model

processor = TrOCRProcessor.from_pretrained('microsoft/trocr-small-handwritten')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-small-handwritten')
pixel_values = processor(images=final_img, return_tensors="pt").pixel_values

generated_ids = model.generate(pixel_values)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


priting the results and concluding that the result of OCR is getting better with the noise removal

In [ ]:
print(generated_text)

clean old Panadiert . From watch


time to fine-tune the model using a dataset consisting of photos of my own handwriting and labels containing the correct text

for this opeartion I'll be using GPU with CUDA

In [ ]:
!nvidia-smi

Tue Apr  7 13:20:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpu759t94y".


importing the dataset by connecting the google drive in which i have uploaded the dataset folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


uploading our model as per usual and getting started to train it using the new dataset

In [ ]:

model_id = "microsoft/trocr-small-handwritten"

processor = TrOCRProcessor.from_pretrained(model_id)
model = VisionEncoderDecoderModel.from_pretrained(model_id)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

The image processor of type `DeiTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-small-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

VisionEncoderDecoderModel(
  (encoder): DeiTModel(
    (embeddings): DeiTEmbeddings(
      (patch_embeddings): DeiTPatchEmbeddings(
        (projection): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DeiTEncoder(
      (layer): ModuleList(
        (0-11): 12 x DeiTLayer(
          (attention): DeiTAttention(
            (attention): DeiTSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
            )
            (output): DeiTSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): DeiTIntermediate(
            (dense): Linear(in_features=384, out_features=1536, bias=True)
        

now, Loading the dataset in python

In [ ]:
MAX_LABEL_LENGTH = 128

class OCRDataset(Dataset):
    def __init__(self, root_dir, processor, max_target_length=MAX_LABEL_LENGTH):
        self.root_dir          = root_dir
        self.processor         = processor
        self.max_target_length = max_target_length

        with open(f"{root_dir}/labels.txt", "r") as f:
            self.samples = [line.strip().split("|") for line in f if "|" in line]

    def preprocess(self, image):
        img       = np.array(image)
        img_gray  = grayscale(img)
        _, img_bw = cv2.threshold(img_gray, 0, 255,
                                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        no_noise  = noise_removal(img_bw)
        return Image.fromarray(no_noise).convert("RGB")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, text = self.samples[idx]

        image = Image.open(f"{self.root_dir}/images/{img_name}").convert("RGB")
        image = self.preprocess(image)

        # Encode image
        pixel_values = self.processor(images=image, return_tensors="pt").pixel_values

        # Tokenize labels with fixed max_length + replace pad with -100
        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids

        # Replace padding token id with -100 so loss ignores it
        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {
            "pixel_values": pixel_values.squeeze(0),
            "labels":       labels.squeeze(0)
        }

In [ ]:
dataset_path = "/content/drive/MyDrive/dataset"
dataset = OCRDataset(dataset_path, processor)

the training loop

In [ ]:
from transformers import get_scheduler

NUM_EPOCHS = 40

loader = DataLoader(dataset, batch_size=8, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=100,
    num_training_steps=len(loader) * NUM_EPOCHS
)

model.train()

for epoch in range(NUM_EPOCHS):
    total_loss = 0.0

    for batch in loader:
        pixel_values = batch["pixel_values"].to(device)
        labels       = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss    = outputs.loss

        loss.backward()
        optimizer.step()
        scheduler.step()  # ← reduce lr gradually

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch + 1} | Avg Loss: {avg_loss:.4f}")

Epoch 1 | Avg Loss: 10.0668
Epoch 2 | Avg Loss: 6.7986
Epoch 3 | Avg Loss: 3.2135
Epoch 4 | Avg Loss: 1.7599
Epoch 5 | Avg Loss: 1.1510
Epoch 6 | Avg Loss: 0.8345
Epoch 7 | Avg Loss: 0.7473
Epoch 8 | Avg Loss: 0.6965
Epoch 9 | Avg Loss: 0.5732
Epoch 10 | Avg Loss: 0.5414
Epoch 11 | Avg Loss: 0.5231
Epoch 12 | Avg Loss: 0.4992
Epoch 13 | Avg Loss: 0.5215
Epoch 14 | Avg Loss: 0.4193
Epoch 15 | Avg Loss: 0.4634
Epoch 16 | Avg Loss: 0.4322
Epoch 17 | Avg Loss: 0.4293
Epoch 18 | Avg Loss: 0.3518
Epoch 19 | Avg Loss: 0.3353
Epoch 20 | Avg Loss: 0.3195
Epoch 21 | Avg Loss: 0.2872
Epoch 22 | Avg Loss: 0.2846
Epoch 23 | Avg Loss: 0.3238
Epoch 24 | Avg Loss: 0.4323
Epoch 25 | Avg Loss: 0.4641
Epoch 26 | Avg Loss: 0.2733
Epoch 27 | Avg Loss: 0.2326
Epoch 28 | Avg Loss: 0.2374
Epoch 29 | Avg Loss: 0.1911
Epoch 30 | Avg Loss: 0.1759
Epoch 31 | Avg Loss: 0.1348
Epoch 32 | Avg Loss: 0.1348
Epoch 33 | Avg Loss: 0.0961
Epoch 34 | Avg Loss: 0.1195
Epoch 35 | Avg Loss: 0.0902
Epoch 36 | Avg Loss: 0.0666


evaluating the model

In [ ]:
def preprocess(image):
    img       = np.array(image)
    img_gray  = grayscale(img)
    _, img_bw = cv2.threshold(img_gray, 0, 255,
                               cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    no_noise  = noise_removal(img_bw)
    return Image.fromarray(no_noise).convert("RGB")

In [ ]:

# Load and preprocess the exact same way as training
image = Image.open("/media/test4.jpg").convert("RGB")
image = preprocess(image)  # ← this was missing before

pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

model.eval()
with torch.no_grad():
    generated_ids = model.generate(
        pixel_values,
        max_new_tokens=128  # ← tell it how long to generate
    )

predicted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
print("Predicted text:", predicted_text[0])

**RQ: models gets the image instance from PIL (data type: PIL image), meanwhile the pre-processing is using CV2 (data type: np array)**

The desired pipeline: PIL → NumPy → OpenCV → NumPy → PIL

Saving the model on Google Collab

In [ ]:
import os

# Create a folder to save into
save_path = "/content/trocr-finetuned"
os.makedirs(save_path, exist_ok=True)

# Save the model and processor
model.save_pretrained(save_path)
processor.save_pretrained(save_path)

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


Saving the fine tuned model on Google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree(
    "/content/trocr-finetuned",
    "/content/drive/MyDrive/trocr-finetuned"
)

print("Saved to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to Google Drive!


Loading back the model from the drive to use it later on

In [ ]:
save_path = "/content/drive/MyDrive/trocr-finetuned"

processor = TrOCRProcessor.from_pretrained(save_path)
model = VisionEncoderDecoderModel.from_pretrained(save_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

VisionEncoderDecoderModel(
  (encoder): DeiTModel(
    (embeddings): DeiTEmbeddings(
      (patch_embeddings): DeiTPatchEmbeddings(
        (projection): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): DeiTEncoder(
      (layer): ModuleList(
        (0-11): 12 x DeiTLayer(
          (attention): DeiTAttention(
            (attention): DeiTSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
            )
            (output): DeiTSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): DeiTIntermediate(
            (dense): Linear(in_features=384, out_features=1536, bias=True)
        

Trying to identify the lines in the images to feed them then to the model

Identifying the general structure

In [ ]:
img_path= '/media/test4.jpg'
image=cv2.imread(img_path)
img_gray_str=grayscale(image)

In [ ]:
deskewed_image= deskew(img_gray_str)
cv2.imwrite('/media/deskewed_image.jpg', deskewed_image)

1


True

In [ ]:
image_blured= cv2.GaussianBlur(deskewed_image,(5,5),0)

In [ ]:
thresh = cv2.threshold(image_blured, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]

kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15,5))

dilate = cv2.dilate(thresh, kernel, iterations=2)

In [ ]:
cnts= cv2.findContours(dilate,cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cnts= cnts[0] if len(cnts) == 2 else cnts[1]
cnts= sorted(cnts, key=lambda x: cv2.boundingRect(x)[0])

image = cv2.imread(img_path)

for c in cnts:
    x, y, w, h = cv2.boundingRect(c)
    if h > 15 and w > 100:  # ← lower the height threshold
        roi = image[y:y+h, x:x+w]
        cv2.rectangle(image, (x, y), (x+w, y+h), (36, 255, 12), 2)

cv2.imwrite('/media/image_contours.jpg', image)

True

the trying to get the structure didn't work so I'll be using another method

In [ ]:

def split_lines(image_path, min_gap=5):
    # Load and threshold
    image = cv2.imread(image_path)
    gray  = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    _, thresh = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    # Count dark pixels per row
    row_sums = np.sum(thresh, axis=1)  # one number per row

    # Find rows that are "empty" (valleys between lines)
    threshold_val = np.max(row_sums) * 0.05  # row must have 5% of max to count as text
    is_text_row   = row_sums > threshold_val  # True = text, False = gap

    # Find where text starts and ends (transitions)
    line_regions = []
    in_line      = False
    start_row    = 0

    for i, is_text in enumerate(is_text_row):
        if is_text and not in_line:
            in_line   = True
            start_row = i
        elif not is_text and in_line:
            in_line = False
            if i - start_row > min_gap:          # ignore tiny noise regions
                line_regions.append((start_row, i))

    # Handle last line if image ends while still in text
    if in_line:
        line_regions.append((start_row, len(is_text_row)))

    # Crop each line and save
    line_images = []
    padding     = 5  # add a few pixels above and below each line

    for i, (y1, y2) in enumerate(line_regions):
        y1  = max(0, y1 - padding)
        y2  = min(image.shape[0], y2 + padding)
        roi = image[y1:y2, :]                    # full width, cropped height

        line_images.append(roi)
        cv2.imwrite(f'/media/line_{i}.jpg', roi)

        # Draw on original for visualization
        cv2.rectangle(image, (0, y1), (image.shape[1], y2), (36, 255, 12), 2)

    cv2.imwrite('/media/image_contours.jpg', image)
    print(f"Found {len(line_images)} lines")
    return line_images


In [ ]:
line_images = split_lines("/media/deskewed_image.jpg")

Found 1 lines


still the split lines method didn't work so I'll be using another OCR model to identify the text instead of trying to do it myself

PaddleOCR was being difficult soo

Trying another model as well called easyOCR

In [ ]:
!pip install easyocr --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 21.3 MB/s eta 0:00:00


In [ ]:
import easyocr

img_path= '/media/test4.jpg'
reader = easyocr.Reader(['en'], gpu=False)

# Detect lines only
result = reader.detect(img_path)
boxes  = result[0][0]  # horizontal boxes

image = cv2.imread(img_path)
model.eval()

for i, box in enumerate(boxes):
    x1, x2, y1, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])

    # Add small padding
    x1 = max(x1 - 5, 0)
    y1 = max(y1 - 5, 0)
    x2 = min(x2 + 5, image.shape[1])
    y2 = min(y2 + 5, image.shape[0])

    roi = image[y1:y2, x1:x2]
    cv2.imwrite(f'/media/line_{i}.jpg', roi)

    # Feed into your TrOCR model
    line_pil = Image.fromarray(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
    line_pil = preprocess(line_pil)

    pixel_values = processor(images=line_pil, return_tensors="pt").pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model.generate(pixel_values, max_new_tokens=128)

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    print(f"Line {i+1}: {text}")

RuntimeError: module compiled against ABI version 0x1000009 but this version of numpy is 0x2000000

ImportError: numpy.core.multiarray failed to import

easyOCR wasn't working as it kept identifying wrong lines and flipping the images

will be using another method to try to get our paragraphs

In [ ]:

def read_paragraph(image_path, processor, model, device,
                   window_height=60, step=40, min_text_density=0.01):

    image     = cv2.imread(image_path)
    gray      = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    h, w      = gray.shape

    _, thresh = cv2.threshold(gray, 0, 255,
                              cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    results = []
    y       = 0

    while y + window_height <= h:
        strip   = thresh[y:y + window_height, :]
        density = np.sum(strip) / (255 * strip.size)

        if density > min_text_density:
            roi      = image[y:y + window_height, :]
            line_pil = Image.fromarray(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
            line_pil = preprocess(line_pil)

            pixel_values = processor(
                images=line_pil, return_tensors="pt"
            ).pixel_values.to(device)

            with torch.no_grad():
                generated_ids = model.generate(pixel_values, max_new_tokens=128)

            text = processor.batch_decode(
                generated_ids, skip_special_tokens=True
            )[0].strip()

            if text:
                results.append((y, text))

        y += step

    # Remove duplicates from overlapping windows
    final_results = []
    seen_texts    = set()

    for y_pos, text in results:
        is_duplicate = any(
            sum(a == b for a, b in zip(text, seen))
            / max(len(text), len(seen)) > 0.8
            for seen in seen_texts
        )
        if not is_duplicate:
            final_results.append(text)
            seen_texts.add(text)

    return final_results

img_path= '/media/test4.jpg'

lines = read_paragraph(img_path, processor, model, device, window_height=90, step=80, min_text_density=0.03)

for i, line in enumerate(lines):
    print(f"Line {i+1}: {line}")

Line 1: and the batch of the proceeds with the


In [ ]:
image     = cv2.imread(img_path)
gray      = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
h, w      = gray.shape

_, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

y = 0
i = 0
while y + 85 <= h:
    strip    = thresh[y:y + 85, :]
    density  = np.sum(strip) / (255 * strip.size)
    roi      = image[y:y + 85, :]

    # Save each strip
    cv2.imwrite(f'/media/strip{i}.jpg', roi)

    y += 85
    i += 1

In [ ]:

# Load and preprocess the exact same way as training
image = Image.open("/media/part_3.jpg").convert("RGB")
image = preprocess(image)  # ← this was missing before

pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)

model.eval()
with torch.no_grad():
    generated_ids = model.generate(
        pixel_values,
        max_new_tokens=128  # ← tell it how long to generate: needs to be more
    )

predicted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
print("Predicted text:", predicted_text[0])

Predicted text: 


In [ ]:
image     = cv2.imread("/media/test4.jpg")
gray      = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
h, w      = gray.shape

_, thresh = cv2.threshold(gray, 0, 255,
                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

y = 0
i = 0
k= h//4

while y + k <= h:
    strip   = thresh[y:y + k, :]
    density = np.sum(strip) / (255 * strip.size)
    roi     = image[y:y + k, :]

    cv2.imwrite(f'/media/part_{i}.jpg', roi)
    print(f"Strip {i}: y={y} to y={y+k} | density={density:.4f}")

    y += k
    i += 1

# Save leftover part at the bottom
if y < h:
    roi     = image[y:h, :]
    density = np.sum(thresh[y:h, :]) / (255 * thresh[y:h, :].size)

    cv2.imwrite(f'/media/part_{i}.jpg', roi)
    print(f"Strip {i}: y={y} to y={h} | density={density:.4f} ← leftover")

Strip 0: y=0 to y=78 | density=0.1952
Strip 1: y=78 to y=156 | density=0.1893
Strip 2: y=156 to y=234 | density=0.1948
Strip 3: y=234 to y=312 | density=0.1230
